In [ ]:
!pip install transformers datasets evaluate huggingface_hub
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is 

In [ ]:
#PARTE 1
import random
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Cargar dataset
dataset = load_dataset("financial_phrasebank", "sentences_allagree")

# Cargar FinBERT
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Armar modelo
nlp_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Diccionario de etiquetas y emojis
label_map = {
    0: ("positive", "🟢"),
    1: ("negative", "🔴"),
    2: ("neutral",  "🟡")
}

# Seleccionar 10 frases aleatorias
random_indices = random.sample(range(len(dataset['train'])), 10)

# Analizar
for idx in random_indices:
    phrase = dataset['train'][idx]['sentence']
    label = dataset['train'][idx]['label']
    pred = nlp_pipeline(phrase)[0]

    true_label_text, true_emoji = label_map[label]
    pred_label_text = pred['label'].lower()
    pred_emoji = {"positive": "🟢", "negative": "🔴", "neutral": "🟡"}[pred_label_text]



    print("──────────────────────────────────────────────────────")
    print(f"📄 Frase      : {phrase}")
    print(f"🏷️  Etiqueta   : {true_emoji} {true_label_text.upper()}")
    print(f"🤖 Predicción : {pred_emoji} {pred['label']} ({pred['score']*100:.2f}%)")


Device set to use cpu


──────────────────────────────────────────────────────
📄 Frase      : The contract value amounts to about EUR11m , the company added .
🏷️  Etiqueta   : 🔴 NEGATIVE
🤖 Predicción : 🟡 neutral (92.23%)
──────────────────────────────────────────────────────
📄 Frase      : `` Small firms are suffering at the moment because they are likely to have money trouble , '' he added .
🏷️  Etiqueta   : 🟢 POSITIVE
🤖 Predicción : 🔴 negative (96.30%)
──────────────────────────────────────────────────────
📄 Frase      : The value of the orders is about EUR 70mn .
🏷️  Etiqueta   : 🔴 NEGATIVE
🤖 Predicción : 🟡 neutral (94.21%)
──────────────────────────────────────────────────────
📄 Frase      : Finnish airline Finnair has won a deal with the UK public sector to be the official airline for flights from London Heathrow to Osaka in Japan , as well as flights between Manchester in the UK and Helsinki in Finland .
🏷️  Etiqueta   : 🟡 NEUTRAL
🤖 Predicción : 🟢 positive (92.71%)
──────────────────────────────────────

In [1]:
!pip install pymupdf transformers sentencepiece --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 16.1 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving Contrato_ejercicio.pdf to Contrato_ejercicio.pdf


In [4]:
#Parte 2
import fitz

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

pdf_path = next(iter(uploaded))
full_text = extract_text_from_pdf(pdf_path)

#  Preparar modelo de resumen
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

#  Dividir texto largo por si es largo
def split_text(text, max_tokens=1024):
    import textwrap
    return textwrap.wrap(text, max_tokens, break_long_words=False)

text_chunks = split_text(full_text)

# Generar resumen por partes
summary_parts = []
for i, chunk in enumerate(text_chunks):
    print(f"🔹 Resumiendo parte {i+1}/{len(text_chunks)}...")
    summary = summarizer(chunk, max_length=75, min_length=30, do_sample=False)[0]['summary_text']
    summary_parts.append(summary)

#  Mostrar resumen final
final_summary = "\n\n".join(summary_parts)
print("\n📄 ✨ Resumen Ejecutivo del Contrato ✨ 📄\n")
print(final_summary)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


🔹 Resumiendo parte 1/11...
🔹 Resumiendo parte 2/11...
🔹 Resumiendo parte 3/11...
🔹 Resumiendo parte 4/11...
🔹 Resumiendo parte 5/11...
🔹 Resumiendo parte 6/11...
🔹 Resumiendo parte 7/11...
🔹 Resumiendo parte 8/11...
🔹 Resumiendo parte 9/11...
🔹 Resumiendo parte 10/11...
🔹 Resumiendo parte 11/11...

📄 ✨ Resumen Ejecutivo del Contrato ✨ 📄

El contrato de Arrendamiento o Alquiler del Local  Comercial que celebran, de una parte ZAMORA QUISPE, MARÍA ELENA,  identificada with domicilio real en Calle Los Cipreses N.º 123, Urb. Jard

Los Laureles N.º 451 and 448, se  encuentra desocupado, en buen estado de conservación y habitabilidad, con piso de  cerámica tipo porcelanato. LA ARRENDADORA, le hace entrega del local comercial en bu

La forma de pago de la renta será por mensualidades, realizándose el abono el día 11 de cada mes. Las partes convienen fijar un plazo de duración determinada para el presente  contrato, el cual será de 12 meses.

EL ARRENDATARIO está obligado a desocupar y devolver

In [7]:
!pip install transformers datasets pymupdf


In [18]:
#Bonus Extracción de Nombres
# 1. Librerías
import re
from collections import defaultdict
from transformers import pipeline
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')


#  Limpiar el texto (quitar símbolos raros, errores de OCR, etc.)
def clean_text(txt):
    txt = re.sub(r'[^A-Za-zÁÉÍÓÚÑáéíóúñ0-9.,;:()\-/ ]+', ' ', txt)
    txt = re.sub(r'\s+', ' ', txt)
    return txt.strip()

cleaned_text = clean_text(final_summary)

# Dividir en oraciones más pequeñas (para mejorar la precisión del NER)
sentences = sent_tokenize(cleaned_text, language='spanish')
chunks = [" ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]  # agrupar de 3 en 3

#  Cargar el modelo de NER en español
ner_pipeline = pipeline("ner", model="mrm8488/bert-spanish-cased-finetuned-ner", grouped_entities=True)

#  Extraer entidades
entities = defaultdict(set)
for chunk in chunks:
    ner_results = ner_pipeline(chunk)
    for ent in ner_results:
        label = ent['entity_group']
        word = ent['word'].strip().upper()
        word = re.sub(r'[^\w\sÁÉÍÓÚÑáéíóúñ.-]', '', word)
        if label and len(word) > 1:
            if label == "PER":
                entities["Personas (PER)"].add(word)
            elif label == "LOC":
                entities["Ubicación (LOC)"].add(word)
            elif label == "ORG":
                entities["Roles (ORG)"].add(word)
            elif label == "MISC":
                entities["Conceptos legales (MISC)"].add(word)

#  Detectar leyes mencionadas explícitamente
leyes_detectadas = set()
for match in re.findall(r"(Código Civil|Constitución|Ley N°\s*\d+|Código Penal)", final_summary, re.IGNORECASE):
    leyes_detectadas.add(match.title())

#  Conceptos legales adicionales típicos si aparecen
conceptos_clave = ['alquiler', 'arrendamiento', 'garantía', 'local comercial', 'contrato', 'plazo', 'patrimonio']
for concepto in conceptos_clave:
    if concepto in final_summary.lower():
        entities["Conceptos legales (MISC)"].add(concepto.upper())

# Mostrar el resultado
print("\n📌 **Entidades encontradas en el resumen del contrato**\n")

for category in ["Personas (PER)", " Roles (ORG)","Ubicación (LOC)", "Conceptos legales (MISC)"]:
    print(f"**{category}**:")
    if entities[category]:
        for item in sorted(entities[category]):
            print(f"- {item}")
    else:
        print("- (No se encontraron)")
    print()

print("**Leyes aplicables (LEY)**:")
if leyes_detectadas:
    for ley in sorted(leyes_detectadas):
        print(f"- {ley}")
else:
    print("- (No se encontraron)")



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
Some weights of the model checkpoint at mrm8488/bert-spanish-cased-finetuned-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and


📌 **Entidades encontradas en el resumen del contrato**

**Personas (PER)**:
- ELENA
- ZAMORA QUISPE

**Ubicación (LOC)**:
- CALLE LOS CIPRESES N. 123
- JARD LOS LAURELES
- LIMA
- UR

**Organizaciones / Roles (ORG)**:
- EL
- EL ARRENDATARI
- EL ARRENDATARIO
- EL ARRENTARA
- EL INQUILINO
- INQUILINO
- LA ARRENDAD
- LA ARRENDADORA
- QUILINO

**Conceptos legales (MISC)**:
- ALQUI
- ALQUILER
- ARRE
- ARRENDAMIENTO
- CONTRATO
- CÓDIGO CIVIL
- FAR
- GARANTÍA
- IN
- LOCAL COMERCIAL
- PLAZO

**Leyes aplicables (LEY)**:
- Código Civil
